In [1]:
import argparse
import os
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=100000
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [4]:
x = pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv")
x = torch.from_numpy(numpy.nan_to_num(x[['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount',
                                         'type_2_discount', 'type_3_discount', 'type_4_discount',  'type_5_discount', 
                                         'type_6_discount']].to_numpy().astype(numpy.float32)))

y=torch.from_numpy(numpy.nan_to_num(pandas.read_csv("/kaggle/input/rohlik-sales-forecasting-challenge-v2/sales_train.csv")['sales'].to_numpy().astype(numpy.float32)))

In [5]:
train_x = torch.utils.data.DataLoader(x,batch_size=batch_size)
train_y = torch.utils.data.DataLoader(y,batch_size=batch_size)

In [6]:
class RF_NET(nn.Module):
    def __init__(self):
        super().__init__()
        self.rafire = nn.Sequential(
            
            nn.Linear(9, 1524),
            nn.Linear(1524, 824),
            nn.Linear(824, 424),
            nn.Linear(424, 124),
            nn.Linear(124, 1)
        )

    def forward(self,x):
        #x,_=self.rafire_0(x)
        #print(x,x.shape)
        return self.rafire(x)


In [7]:
rf_net = RF_NET().type(torch.float32).to(device)
rf_net= nn.DataParallel(rf_net).to(device)
#rf_net.load_state_dict(torch.load("/kaggle/input/encoder-weight-data/weight.pth",weights_only=False,map_location=torch.device('cpu')))

In [8]:
torch.save(rf_net.state_dict(),f'/kaggle/working/weight.pth')